# Firecrawl + Moss Cookbook

Crawl one or more URLs with Firecrawl, convert the results to clean markdown, index them into Moss, and query the knowledge base semantically.

## 1. Set Up Project Environment

Install the SDKs and set your credentials before running the notebook.

In [1]:
#pip install firecrawl-py moss python-dotenv

In [2]:
from __future__ import annotations

import os
import uuid
from dataclasses import dataclass
from typing import Any

from dotenv import load_dotenv
from firecrawl import Firecrawl
from moss import DocumentInfo, MossClient, QueryOptions

load_dotenv()

FIRECRAWL_API_KEY = os.getenv("FIRECRAWL_API_KEY")
MOSS_PROJECT_ID = os.getenv("MOSS_PROJECT_ID")
MOSS_PROJECT_KEY = os.getenv("MOSS_PROJECT_KEY")
DEFAULT_QUERY = "What does the knowledge base say about the topic?"

## 2. Define Core Data Structures

Normalize each crawled page into a small Python structure before converting it into Moss documents.

In [3]:
@dataclass
class CrawledPage:
    url: str
    markdown: str
    title: str | None = None


def page_to_crawled_page(page: Any) -> CrawledPage:
    markdown = getattr(page, "markdown", None)
    if markdown is None and isinstance(page, dict):
        markdown = page.get("markdown")

    metadata = getattr(page, "metadata", None)
    if metadata is None and isinstance(page, dict):
        metadata = page.get("metadata", {})

    url = None
    title = None
    if isinstance(metadata, dict):
        url = metadata.get("source_url") or metadata.get("sourceURL") or metadata.get("url")
        title = metadata.get("title") or metadata.get("og_title")
    elif metadata is not None:
        url = getattr(metadata, "source_url", None) or getattr(metadata, "sourceURL", None) or getattr(metadata, "url", None)
        title = getattr(metadata, "title", None) or getattr(metadata, "og_title", None)

    return CrawledPage(url=url or "unknown", markdown=markdown or "", title=title)


def crawled_pages_to_moss_docs(pages: list[CrawledPage]) -> list[DocumentInfo]:
    docs: list[DocumentInfo] = []
    for index, page in enumerate(pages, start=1):
        docs.append(
            DocumentInfo(
                id=f"firecrawl-{index}",
                text=page.markdown,
                metadata={"source_url": page.url, "title": page.title or ""},
            )
        )
    return docs

## 3. Implement Main Functionality

Firecrawl handles URL-to-markdown extraction. Moss handles indexing and semantic search.

In [ ]:
def validate_configuration(urls: list[str]) -> None:
    if not urls:
        raise ValueError("Provide at least one URL to crawl.")
    if not FIRECRAWL_API_KEY:
        raise ValueError("Set FIRECRAWL_API_KEY before running the notebook.")
    if not MOSS_PROJECT_ID or not MOSS_PROJECT_KEY:
        raise ValueError("Set MOSS_PROJECT_ID and MOSS_PROJECT_KEY before running the notebook.")


def crawl_urls(urls: list[str], limit: int = 3) -> list[CrawledPage]:
    firecrawl = Firecrawl(api_key=FIRECRAWL_API_KEY)
    pages: list[CrawledPage] = []

    for url in urls:
        job = firecrawl.crawl(url=url, limit=limit, scrape_options={"formats": ["markdown"]})
        raw_pages = getattr(job, "data", None) or (job.get("data") if isinstance(job, dict) else []) or []
        pages.extend(page_to_crawled_page(page) for page in raw_pages)

    return [page for page in pages if page.markdown.strip()]


async def prepare_knowledge_base(urls: list[str], limit: int = 10) -> tuple[MossClient, str]:
    validate_configuration(urls)
    crawled_pages = crawl_urls(urls, limit=limit)
    documents = crawled_pages_to_moss_docs(crawled_pages)

    if not documents:
        raise RuntimeError("Firecrawl returned no markdown content to index.")

    index_name = f"firecrawl-cookbook-{uuid.uuid4().hex[:8]}"
    client = MossClient(MOSS_PROJECT_ID, MOSS_PROJECT_KEY)

    await client.create_index(index_name, documents)
    await client.load_index(index_name)

    print(f"Indexed {len(documents)} documents into {index_name}")
    return client, index_name


async def query_knowledge_base(client: MossClient, index_name: str, query: str = DEFAULT_QUERY) -> None:
    results = await client.query(index_name, query, QueryOptions(top_k=3, alpha=0.8))

    print(f"Query: {query}")
    for item in results.docs:
        source_url = item.metadata.get("source_url", "unknown") if item.metadata else "unknown"
        print(f"- [{item.score:.3f}] {source_url}")
        print(f"  {item.text[:200].strip()}")


# Build knowledge base and query it in one step
async def build_and_query_knowledge_base(urls: list[str], query: str = DEFAULT_QUERY) -> None:
    client, index_name = await prepare_knowledge_base(urls)
    await query_knowledge_base(client, index_name, query)

## 4. Full Firecrawl + Moss Test (Crawl, Index, and Query)


Enter URLs and a question to run end-to-end Firecrawl ingestion and Moss semantic search.

In [5]:
urls = ["https://docs.moss.dev"]

# Crawl + index once
client, index_name = await prepare_knowledge_base(urls)

Indexed 10 documents into firecrawl-cookbook-af681b7b


In [6]:
# Query multiple times without crawling again
await query_knowledge_base(client, index_name, "What is Moss used for?")

Query: What is Moss used for?
- [1.000] https://docs.moss.dev/docs/start/what-is-moss
  [Skip to main content](https://docs.moss.dev/docs/start/what-is-moss#content-area)

[Moss Docs home page![light logo](https://mintcdn.com/moss-afcfb0b6/b460p8xEydp14WML/logo/moss-wordmark-light.svg?fi
- [0.939] https://docs.moss.dev/docs/api-reference/v1/getting-started/introduction
  [Skip to main content](https://docs.moss.dev/docs/api-reference/v1/getting-started/introduction#content-area)

[Moss Docs home page![light logo](https://mintcdn.com/moss-afcfb0b6/b460p8xEydp14WML/logo
- [0.912] https://docs.moss.dev/docs/reference/python/interfaces/JobStatus
  [Skip to main content](https://docs.moss.dev/docs/reference/python/interfaces/JobStatus#content-area)

[Moss Docs home page![light logo](https://mintcdn.com/moss-afcfb0b6/b460p8xEydp14WML/logo/moss-wo


In [8]:
await query_knowledge_base(
    client,
    index_name,
    "What evidence in the docs supports the claim of sub-10 ms search, and what assumptions or caveats should an engineering team validate before adoption?",
)

Query: What evidence in the docs supports the claim of sub-10 ms search, and what assumptions or caveats should an engineering team validate before adoption?
- [0.952] https://docs.moss.dev/docs/start/what-is-moss
  [Skip to main content](https://docs.moss.dev/docs/start/what-is-moss#content-area)

[Moss Docs home page![light logo](https://mintcdn.com/moss-afcfb0b6/b460p8xEydp14WML/logo/moss-wordmark-light.svg?fi
- [0.907] https://docs.moss.dev/docs/api-reference/v1/document-operations/getDocs
  [Skip to main content](https://docs.moss.dev/docs/api-reference/v1/document-operations/getDocs#content-area)

[Moss Docs home page![light logo](https://mintcdn.com/moss-afcfb0b6/b460p8xEydp14WML/logo/
- [0.891] https://docs.moss.dev/docs/api-reference/v1/document-operations/deleteDocs
  [Skip to main content](https://docs.moss.dev/docs/api-reference/v1/document-operations/deleteDocs#content-area)

[Moss Docs home page![light logo](https://mintcdn.com/moss-afcfb0b6/b460p8xEydp14WML/lo
